# 基于双向 GRU (Bi-GRU) 与全局池化的电影评论情感分析 (全量数据版)

本 Notebook 演示如何使用 PyTorch 搭建一个高性能、带**全局最大池化 (Global Max Pooling)** 机制与 **早停 (Early Stopping)** 的双向 GRU (Bi-GRU) 文本情感分类系统。

### 🌟 本版本的核心突破与改进：
1. **网络架构**：采用 **Bi-GRU** 捕捉文本特征上下文信息。
2. **引入全局最大池化 (Global Max Pooling)**：通过 `torch.max(gru_out, dim=1)[0]` 提取文本全长中最强烈的语义信号，杜绝尾部 `<PAD>` 填充符干扰。
3. **扩充至全量数据容量**：开启全量数据集加载（`sample_per_class=None`），提供丰富多样的文本特征表达，防止小样本拟合盲区。
4. **验证集 (85:15) 与 Early Stopping**：实时监控 `val_loss`，触发早停时保存并还原最佳权重 `best_gru_model.pt`。

## 1. 导入必要的 Python 库与全量均衡数据采样

In [1]:
import os
import re
import collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/Colab_Data/RNN评论情感分析/')

# 设置随机种子确保结果可复现
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

def load_balanced_data(file_path, sample_per_class=None, random_seed=42):
    df = pd.read_csv(file_path)
    df_pos = df[df['真实标签'] == 1]
    df_neg = df[df['真实标签'] == 0]
    if sample_per_class is not None:
        pos_sampled = df_pos.sample(n=min(sample_per_class, len(df_pos)), random_state=random_seed)
        neg_sampled = df_neg.sample(n=min(sample_per_class, len(df_neg)), random_state=random_seed)
    else:
        # 全量均衡加载
        min_len = min(len(df_pos), len(df_neg))
        pos_sampled = df_pos.sample(n=min_len, random_state=random_seed)
        neg_sampled = df_neg.sample(n=min_len, random_state=random_seed)
    return pd.concat([pos_sampled, neg_sampled]).sample(frac=1, random_state=random_seed).reset_index(drop=True)

train_path = 'data/data_train.csv'
test_path = 'data/data_test.csv'

# sample_per_class=None 全量数据加载
df_raw_train = load_balanced_data(train_path, sample_per_class=None)
df_test = load_balanced_data(test_path, sample_per_class=None)

# 按 85:15 划分 Train 集与 Validation 集
df_train, df_val = train_test_split(df_raw_train, test_size=0.15, random_state=42, stratify=df_raw_train['真实标签'])

print(f"训练集样本数量: {len(df_train)} (正面: {sum(df_train['真实标签']==1)}, 负面: {sum(df_train['真实标签']==0)})")
print(f"验证集样本数量: {len(df_val)}   (正面: {sum(df_val['真实标签']==1)}, 负面: {sum(df_val['真实标签']==0)})")
print(f"测试集样本数量: {len(df_test)}  (正面: {sum(df_test['真实标签']==1)}, 负面: {sum(df_test['真实标签']==0)})")
display(df_train.head())

Mounted at /content/drive
训练集样本数量: 21250 (正面: 10625, 负面: 10625)
验证集样本数量: 3750   (正面: 1875, 负面: 1875)
测试集样本数量: 25000  (正面: 12500, 负面: 12500)


,影评内容,真实标签
11904,I only watched the first 30 minutes of this an...,0
19929,Its no surprise that Busey later developed a t...,1
15180,This is the second movie I saw for Horrorfest ...,0
12711,"i read the book ""7 years in Tibet"" from Heinri...",0
4295,I don't know what the Oscar voters saw in this...,0


## 2. 文本清洗、词表构建与预训练词向量加载

In [2]:
def clean_text(text):
    if not isinstance(text, str):
        return []
    # 去除 HTML 标签 <br />
    text = re.sub(r'<br\s*/?>', ' ', text)
    # 仅保留英文字母
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = text.lower().split()
    return tokens

train_tokens = [clean_text(text) for text in df_train['影评内容']]
val_tokens = [clean_text(text) for text in df_val['影评内容']]
test_tokens = [clean_text(text) for text in df_test['影评内容']]

def build_vocab(tokenized_texts, max_vocab_size=15000):
    counter = collections.Counter()
    for tokens in tokenized_texts:
        counter.update(tokens)

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_vocab_size - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_tokens, max_vocab_size=15000)
print(f"词表构建完成，限制上限大小为: {len(vocab)}")

def load_pretrained_embeddings(vocab, embed_dim=128, glove_path=None):
    vocab_size = len(vocab)
    embedding_matrix = np.random.normal(scale=1.0 / np.sqrt(embed_dim), size=(vocab_size, embed_dim))
    embedding_matrix[vocab['<PAD>']] = np.zeros(embed_dim)

    if glove_path and os.path.exists(glove_path):
        print(f"正在加载 GloVe 词向量: {glove_path}...")
        loaded_count = 0
        with open(glove_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                word = parts[0]
                if word in vocab:
                    vector = np.array(parts[1:], dtype=np.float32)
                    if len(vector) == embed_dim:
                        embedding_matrix[vocab[word]] = vector
                        loaded_count += 1
        print(f"成功加载 {loaded_count}/{vocab_size} 个预训练词向量。")
    else:
        print("未找到本地 GloVe 词向量，采用符合标准缩放高斯分布的方差矩阵初始化。")
    return torch.tensor(embedding_matrix, dtype=torch.float32)

pretrained_weight = load_pretrained_embeddings(vocab, embed_dim=128, glove_path='data/glove.6B.100d.txt')

词表构建完成，限制上限大小为: 15000
未找到本地 GloVe 词向量，采用符合标准缩放高斯分布的方差矩阵初始化。


## 3. 序列 Padding 与 PyTorch Dataset / DataLoader 构建

In [3]:
def encode_and_pad(tokens, vocab, max_len=200):
    seq = [vocab.get(word, vocab['<UNK>']) for word in tokens]
    if len(seq) < max_len:
        seq = seq + [vocab['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

class MovieReviewDataset(Dataset):
    def __init__(self, tokenized_texts, labels, vocab, max_len=200):
        self.samples = [encode_and_pad(tokens, vocab, max_len) for tokens in tokenized_texts]
        self.labels = labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return torch.tensor(self.samples[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float32)

max_len = 200
batch_size = 64

train_dataset = MovieReviewDataset(train_tokens, df_train['真实标签'].values, vocab, max_len)
val_dataset = MovieReviewDataset(val_tokens, df_val['真实标签'].values, vocab, max_len)
test_dataset = MovieReviewDataset(test_tokens, df_test['真实标签'].values, vocab, max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Train batches: 333 | Val batches: 59 | Test batches: 391


## 4. 构建双向 GRU + 全局最大池化网络模型

In [4]:
class SentimentBiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.5, pad_idx=0, pretrained_weight=None):
        super(SentimentBiGRU, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if pretrained_weight is not None:
            self.embedding.weight.data.copy_(pretrained_weight)

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        gru_out, _ = self.gru(embedded)

        # 🌟 关键点：使用全局最大池化 (Global Max Pooling)
        # 抽取 200 个时间步上正向与反向的最强语义，规避末端 <PAD> 影响
        out_pooled = torch.max(gru_out, dim=1)[0]

        out = self.dropout(out_pooled)
        logits = self.fc(out).squeeze(1)
        return logits

# 别名兼容，支持名称调用
SentimentGRU = SentimentBiGRU

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SentimentBiGRU(vocab_size=len(vocab), embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.5, pretrained_weight=pretrained_weight)
model.to(device)
print(model)

# 因为 bidirectional=True, num_layers=2，参数会更多
for name, param in model.gru.named_parameters():
    print(name, param.shape)

SentimentBiGRU(
  (embedding): Embedding(15000, 128, padding_idx=0)
  (gru): GRU(128, 128, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)
weight_ih_l0 torch.Size([384, 128])
weight_hh_l0 torch.Size([384, 128])
bias_ih_l0 torch.Size([384])
bias_hh_l0 torch.Size([384])
weight_ih_l0_reverse torch.Size([384, 128])
weight_hh_l0_reverse torch.Size([384, 128])
bias_ih_l0_reverse torch.Size([384])
bias_hh_l0_reverse torch.Size([384])
weight_ih_l1 torch.Size([384, 256])
weight_hh_l1 torch.Size([384, 128])
bias_ih_l1 torch.Size([384])
bias_hh_l1 torch.Size([384])
weight_ih_l1_reverse torch.Size([384, 256])
weight_hh_l1_reverse torch.Size([384, 128])
bias_ih_l1_reverse torch.Size([384])
bias_hh_l1_reverse torch.Size([384])


## 5. Early Stopping 监控与训练循环

In [5]:
class EarlyStopping:
    def __init__(self, patience=4, verbose=True, save_path='data/best_gru_model.pt'):
        self.patience = patience
        self.verbose = verbose
        self.save_path = save_path
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping 计数: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        os.makedirs(os.path.dirname(self.save_path), exist_ok=True)
        torch.save(model.state_dict(), self.save_path)
        if self.verbose:
            print(f"验证集 Loss 改善，已保存最佳权重至 {self.save_path}")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
early_stopping = EarlyStopping(patience=4, verbose=True, save_path='data/best_gru_model.pt')

epochs = 20
for epoch in range(1, epochs + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        train_correct += (preds == targets).sum().item()
        train_total += targets.size(0)

    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)
            val_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")

    early_stopping(epoch_val_loss, model)
    if early_stopping.early_stop:
        print("触发 Early Stopping 条件，终止训练！")
        break

Epoch 01/20 | Train Loss: 0.5061 | Train Acc: 71.91% | Val Loss: 0.3394 | Val Acc: 85.20%
验证集 Loss 改善，已保存最佳权重至 data/best_gru_model.pt
Epoch 02/20 | Train Loss: 0.2758 | Train Acc: 89.15% | Val Loss: 0.3319 | Val Acc: 86.32%
验证集 Loss 改善，已保存最佳权重至 data/best_gru_model.pt
Epoch 03/20 | Train Loss: 0.2072 | Train Acc: 92.10% | Val Loss: 0.3137 | Val Acc: 86.96%
验证集 Loss 改善，已保存最佳权重至 data/best_gru_model.pt
Epoch 04/20 | Train Loss: 0.1614 | Train Acc: 94.07% | Val Loss: 0.4070 | Val Acc: 84.80%
EarlyStopping 计数: 1/4
Epoch 05/20 | Train Loss: 0.1236 | Train Acc: 95.48% | Val Loss: 0.3591 | Val Acc: 86.05%
EarlyStopping 计数: 2/4
Epoch 06/20 | Train Loss: 0.0985 | Train Acc: 96.64% | Val Loss: 0.4298 | Val Acc: 86.27%
EarlyStopping 计数: 3/4
Epoch 07/20 | Train Loss: 0.0804 | Train Acc: 97.26% | Val Loss: 0.4662 | Val Acc: 85.36%
EarlyStopping 计数: 4/4
触发 Early Stopping 条件，终止训练！


## 6. 加载最佳模型并在测试集上进行终极评估

In [6]:
model.load_state_dict(torch.load('data/best_gru_model.pt'))
model.eval()

test_preds = []
test_probs = []
true_labels = df_test['真实标签'].values

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        logits = model(inputs)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        test_probs.extend(probs)
        test_preds.extend(preds)

acc = accuracy_score(true_labels, test_preds)
prec, rec, f1, _ = precision_recall_fscore_support(true_labels, test_preds, average='binary')
cm = confusion_matrix(true_labels, test_preds)

print("=== 测试集终极评估结果 (Bi-GRU + Max Pooling) ===")
print(f"准确率 (Accuracy) : {acc*100:.2f}%")
print(f"精确率 (Precision): {prec*100:.2f}%")
print(f"召回率 (Recall)   : {rec*100:.2f}%")
print(f"F1 得分 (F1 Score): {f1*100:.2f}%")
print("\n混淆矩阵 (Confusion Matrix):")
print(cm)

=== 测试集终极评估结果 (Bi-GRU + Max Pooling) ===
准确率 (Accuracy) : 85.20%
精确率 (Precision): 84.25%
召回率 (Recall)   : 86.58%
F1 得分 (F1 Score): 85.40%

混淆矩阵 (Confusion Matrix):
[[10477  2023]
 [ 1677 10823]]


## 7. 导出预测结果至 CSV 文件

In [7]:
df_result = df_test.copy()
df_result['预测标签'] = test_preds
df_result['预测概率'] = np.round(test_probs, 4)

output_csv = 'data/result_prediction_gru.csv'
df_result.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"预测结果已成功写入 {output_csv}")
display(df_result.head(10))

预测结果已成功写入 data/result_prediction_gru.csv


,影评内容,真实标签,预测标签,预测概率
0,I don't even like watching those late night ta...,1,1,0.6025
1,Jude law gives Keanu Reeves a run for his mone...,0,0,0.1146
2,I am a big fan of this film and found the TV m...,1,1,0.8919
3,It's dreadful rubbish. I liked 'How Do You Wan...,0,0,0.0083
4,"Ron Howard and his ""editors"" only had one job ...",0,1,0.5329
5,I just got back from the GLBT Film Festival at...,1,1,0.8991
6,John Huston made many remarkable and memorable...,1,1,0.8227
7,I want the 99 minutes of my life back that was...,0,0,0.0179
8,You looking for a comic drama with suspense an...,1,1,0.9280
9,I caught this on local Mexican television at 2...,1,1,0.9466


## 8. 单条电影评论情感预测 Demo

In [8]:
def predict_sentiment(review_text, model, vocab, max_len=200):
    model.eval()
    tokens = clean_text(review_text)
    encoded = encode_and_pad(tokens, vocab, max_len)
    tensor_input = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        logit = model(tensor_input)
        prob = torch.sigmoid(logit).item()
        sentiment = "正面评价 (Positive)" if prob >= 0.5 else "负面评价 (Negative)"

    print(f"评论内容: {review_text}")
    print(f"情感预测: {sentiment} (正面置信度: {prob*100:.2f}%)")

sample_review_1 = "This movie was fantastic! The acting was brilliant and the story kept me on the edge of my seat."
predict_sentiment(sample_review_1, model, vocab)

print("-" * 50)
sample_review_2 = "Terrible movie, waste of time. The plot was non-existent and performance was super boring."
predict_sentiment(sample_review_2, model, vocab)

print("-" * 50)
sample_review_3 = "What an incredible movie! I'll watch it again with my friends."
predict_sentiment(sample_review_3, model, vocab)

print("-" * 50)
sample_review_4 = "Although the story is good, the images are terrible."
predict_sentiment(sample_review_4, model, vocab)

print("-" * 50)
sample_review_5 = "I would never watch this movie for the second time."
predict_sentiment(sample_review_5, model, vocab)

评论内容: This movie was fantastic! The acting was brilliant and the story kept me on the edge of my seat.
情感预测: 正面评价 (Positive) (正面置信度: 98.14%)
--------------------------------------------------
评论内容: Terrible movie, waste of time. The plot was non-existent and performance was super boring.
情感预测: 负面评价 (Negative) (正面置信度: 0.90%)
--------------------------------------------------
评论内容: What an incredible movie! I'll watch it again with my friends.
情感预测: 正面评价 (Positive) (正面置信度: 92.53%)
--------------------------------------------------
评论内容: Although the story is good, the images are terrible.
情感预测: 负面评价 (Negative) (正面置信度: 28.08%)
--------------------------------------------------
评论内容: I would never watch this movie for the second time.
情感预测: 正面评价 (Positive) (正面置信度: 58.31%)
